# 03 - Signals and trades, up close

What a strategy sees and does in a window of the TEST block: price with every entry, exit, take-profit and stop;
the three horizons' P(up) against the entry lines; confidence and strength; predicted sigma with the variance
spikes; equity and drawdown, all as the window's own numbers. Set `START` to a bar number, or to
"steepest_fall" / "worst_trade" / "last", and `BARS` to the window length.

In [1]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
STRATEGY = "calibrated_quantile"
START = "worst_trade"           # a bar number, or "steepest_fall" / "worst_trade" / "last"
BARS = 600

In [2]:
import os
from pathlib import Path

from IPython.display import display

from neural_trade.notebook import BacktestExplorer, pick_run
from neural_trade.visualization.trading_dashboard import detail_window

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
explorer = BacktestExplorer.from_run(run_dir, csv_path=CSV_PATH)
res = explorer.run(STRATEGY, costs={"random_seeds": 0}, baselines=False)
start = detail_window(res, BARS, around=START)[0] if isinstance(START, str) else int(START)
explorer.dashboard(res, start=start, end=start + BARS).show()

run: ..\runs\20260924T182915Z-1aeff1c-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T182915Z-1aeff1c-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


## The signal features over the whole block

Shares of bars for the boolean flags, consensus and horizon votes, and the distribution of the numeric features.

In [3]:
for name, table in explorer.signal_summary().items():
    print(name)
    display(table.round(4))

features


,count,mean,std,min,25%,50%,75%,max
weighted_direction,7236.0,0.4969,0.0194,0.3461,0.4874,0.4979,0.5083,0.5912
weighted_move_$,7236.0,-0.1741,1.9098,-10.6495,-1.0101,0.0188,0.6116,14.2097
strength,7236.0,0.0495,0.0304,0.0015,0.0280,0.0419,0.0635,0.3390
avg_confidence,7236.0,0.4975,0.1680,0.0691,0.3563,0.5196,0.6490,0.7640
agreement,7236.0,0.3481,0.0712,0.3333,0.3333,0.3333,0.3333,1.0000
volatility_$,7236.0,219.1576,54.7364,134.5202,171.2936,209.5383,261.3107,412.7095


flags (share of bars true)


,share true,bars,note
magnitude_coherent,0.0000,7236,served delta is 0 on h1 (beta = 0): every comp...
direction_aligned,0.2109,7236,served delta is 0 on h1 (beta = 0): there it o...
var_spike,0.0077,7236,


consensus


,share of bars
up,0.0900
down,0.1488
neutral,0.7612


horizon votes (P > 0.55 or < 0.45)


,share of bars
0 votes,0.7246
1 vote,0.1962
2+ votes,0.0792


confidence scale


,var_scale
confidence = exp(-var / var_scale); var_scale = median predicted variance on the CAL block,1.0546


## Trades

In [4]:
explorer.trade_analytics(res).show()
trades = res.trades_frame()
if len(trades):
    display(trades.groupby("exit_reason")[["net_pnl", "gross_pnl", "bars_held"]].agg(["count", "mean", "sum"]).round(2))
    display(trades.groupby("side")[["net_pnl", "gross_pnl"]].agg(["count", "sum", "mean"]).round(2))
trades.tail(30).round(2)

net_pnl                 gross_pnl                bars_held         \
              count   mean      sum     count   mean     sum     count   mean   
exit_reason                                                                     
REV             107 -17.62 -1885.53       107   4.16  444.89       107   6.47   
SL                5 -61.83  -309.14         5 -40.68 -203.39         5   8.80   
TIME             35 -22.62  -791.62        35  -0.41  -14.35        35  15.00   

                  
             sum  
exit_reason       
REV          692  
SL            44  
TIME         525

net_pnl                 gross_pnl              
        count      sum   mean     count     sum  mean
side                                                 
LONG       98 -1985.14 -20.26        98  174.84  1.78
SHORT      49 -1001.15 -20.43        49   52.31  1.07

,side,entry_bar,exit_bar,entry_price,exit_price,notional,gross_pnl,costs,exit_reason,tp,sl,entry_reason,net_pnl,bars_held,return_pct
117,SHORT,3973,3980,102906.91,102828.29,7569.65,10.32,19.67,REV,None,103437.96,quantile,-9.35,7,-0.12
118,SHORT,3985,3993,102739.40,103083.59,7560.31,-20.78,19.68,REV,None,103359.27,quantile,-40.46,8,-0.54
119,LONG,3995,4002,103041.80,102984.26,7519.84,0.31,19.55,REV,None,102494.84,quantile,-19.24,7,-0.26
120,LONG,4046,4051,103027.75,103019.08,7500.60,3.87,19.51,REV,None,102593.68,quantile,-15.64,5,-0.21
121,LONG,4133,4146,103185.36,102730.32,7484.97,-28.54,19.42,SL,None,102761.15,quantile,-47.96,13,-0.64
122,LONG,4147,4152,102795.40,102713.99,7437.01,-1.43,19.33,REV,None,102234.19,quantile,-20.76,5,-0.28
123,LONG,4397,4412,102359.39,102314.72,7416.24,1.21,19.28,TIME,None,101917.90,quantile,-18.07,15,-0.24
124,SHORT,4633,4648,102032.46,101938.11,7398.17,11.27,19.22,TIME,None,102617.49,quantile,-7.95,15,-0.11
125,SHORT,4649,4664,101891.90,101871.43,7390.23,5.92,19.21,TIME,None,102591.53,quantile,-13.29,15,-0.18
126,SHORT,4789,4798,101650.91,101747.58,7376.94,-2.59,19.18,REV,None,102069.85,quantile,-21.77,9,-0.30
